In [0]:
import pyspark.sql.functions as f

In [0]:
%run ../utility/read_write_util

In [0]:
df = spark.read.table('nyc_cleansed.weather_data')

In [0]:
rename_cols = {
    'location':'location_name',
    'temperature_c': 'temperature',
    'humidity_pct': 'humidity',
    'precipitation_mm': 'precipitation',
    'wind_speed_kmh': 'wind_speed'
}

for key,val in rename_cols.items():
    df = df.withColumnRenamed(key,val)

select_cols = [
    'location_name',
    'date_time',
    'temperature',
    'humidity',
    'precipitation',
    'wind_speed'
]

df = df.select(*select_cols)

In [0]:
display(df)

Read Config & Select only those records

In [0]:
import json 

cities_filter = read_config('./configs/weather_data_filter.json')['cities_to_filter']

In [0]:
df = df.filter(f"location_name in ({', '.join([f'\'{loc}\'' for loc in cities_filter])})")

In [0]:
source_df = df 

Implement SCD - 1

In [0]:
from delta.tables import *

try:

    if not spark.catalog.tableExists("nyc_enterprise.weather_data"):
        raise ValueError(f"Error: The Delta table nyc_enterprise.weather_data was not found.")

    target_df = DeltaTable.forName(spark, 'nyc_enterprise.weather_data')
    print('Target Table Exists & Successfully loaded into dataframe ')

except Exception as e:
    print(e)
    print('Error While Reading , Creating New Table ')
    target_df = spark.createDataFrame([], schema=source_df.schema)
    target_df = target_df.withColumn('created_by', f.lit('manual')) \
                         .withColumn('created_datetime', f.current_timestamp()) \
                         .withColumn('modified_by', f.lit('')) \
                         .withColumn('modified_datetime', f.lit(''))

    spark.sql('CREATE SCHEMA IF NOT EXISTS nyc_enterprise')
    write_data(
        df=target_df,
        target_file_path="nyc_enterprise.weather_data",
        target_file_format='delta',
        mode_type='overwrite'
    )

source_df = source_df.withColumn('created_by', f.lit('data-pipeline')) \
                     .withColumn('created_datetime', f.current_timestamp()) \
                     .withColumn('modified_by', f.lit('')) \
                     .withColumn('modified_datetime', f.lit(''))

from pyspark.sql.functions import col

target_table = DeltaTable.forName(spark, 'nyc_enterprise.weather_data')

target_table.alias('target').merge(
    source_df.alias('source'),
    'target.location_name = source.location_name AND target.date_time = source.date_time'
).whenMatchedUpdate(
    condition=(
        'target.temperature <> source.temperature OR '
        'target.humidity <> source.humidity OR '
        'target.precipitation <> source.precipitation OR '
        'target.wind_speed <> source.wind_speed'
    ),
    set={
        "temperature": col("source.temperature"),
        "humidity": col("source.humidity"),
        "precipitation": col("source.precipitation"),
        "wind_speed": col("source.wind_speed"),
        "modified_by": f.lit('data-pipeline'),
        "modified_datetime": f.current_timestamp()
    }
).whenNotMatchedInsert(
    values={
        "location_name": col("source.location_name"),
        "date_time": col("source.date_time"),
        "temperature": col("source.temperature"),
        "humidity": col("source.humidity"),
        "precipitation": col("source.precipitation"),
        "wind_speed": col("source.wind_speed"),
        "created_by": f.lit('data-pipeline'),
        "created_datetime": f.current_timestamp(),
        "modified_by": f.lit(''),
        "modified_datetime": f.lit('')
    }
).execute()

print('Updated the Weather Table')